In [1]:
import os
import pandas as pd

# List files in the current working directory
print("Current directory:", os.getcwd())
print("\nAvailable files:")

for file in os.listdir():
    print(file)



df = pd.read_csv("../data/Sample - Superstore.csv", encoding="cp1252")

print("Dataset loaded successfully")
print("Shape:", df.shape)

display(df.head())

Current directory: d:\consulting\Retail Profitability Turnaround\notebook

Available files:
01_load_data.ipynb
02_analyis.ipynb
03_Root-Cause Analysis of Profitability Gaps.ipynb
Dataset loaded successfully
Shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## Analyze profitability by discount level

In [2]:
discount_analysis = (
    df.groupby("Discount")
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Records=("Profit", "size"),
        Loss_Making_Records=(
            "Profit", lambda x: (x < 0).sum()
        )
    )
    .reset_index()
)

discount_analysis["Profit_Margin_%"] = (
    discount_analysis["Profit"]
    / discount_analysis["Sales"] * 100
)

discount_analysis["Loss_Record_Share_%"] = (
    discount_analysis["Loss_Making_Records"]
    / discount_analysis["Records"] * 100
)

display(discount_analysis.round(2))

,Discount,Sales,Profit,Records,Loss_Making_Records,Profit_Margin_%,Loss_Record_Share_%
0,0.00,1087908.47,320987.60,4798,0,29.51,0.00
1,0.10,54369.35,9029.18,94,4,16.61,4.26
2,0.15,27558.52,1418.99,52,17,5.15,32.69
3,0.20,764594.37,90337.31,3657,502,11.82,13.73
4,0.30,103226.66,-10369.28,227,208,-10.05,91.63
5,0.32,14493.46,-2391.14,27,27,-16.50,100.00
6,0.40,116417.78,-23057.05,206,180,-19.81,87.38
7,0.45,5484.97,-2493.11,11,11,-45.45,100.00
8,0.50,58918.54,-20506.43,66,66,-34.80,100.00
9,0.60,6644.70,-5944.66,138,138,-89.46,100.00


Three insights worth reporting
1. The high-discount transactions are a material concern.
The 30%–80% discount groups collectively generate negative profit. These groups warrant investigation before management continues similar discounting practices.
2. The relationship is not perfectly monotonic at low discounts.
The 15% discount group has a lower margin than the 20% group. This reminds us that discount alone may not explain profitability: products, quantities, order characteristics and customer mix may differ.
3. The 70% and 80% groups have particularly severe negative margins.
Their aggregate losses are substantial relative to their sales. However, we must check which products and subcategories appear in these groups before proposing a policy change.

## Investigate Tables and Bookcases

In [3]:
target_products = df[
    df["Sub-Category"].isin(["Tables", "Bookcases"])
]

product_discount_analysis = (
    target_products
    .groupby(["Sub-Category", "Discount"])
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Records=("Profit", "size")
    )
    .reset_index()
)

product_discount_analysis["Profit_Margin_%"] = (
    product_discount_analysis["Profit"]
    / product_discount_analysis["Sales"] * 100
)

product_discount_analysis["Loss_Record_Share_%"] = (
    target_products
    .groupby(["Sub-Category", "Discount"])["Profit"]
    .apply(lambda x: (x < 0).mean() * 100)
    .values
)

display(product_discount_analysis.round(2))

,Sub-Category,Discount,Sales,Profit,Records,Profit_Margin_%,Loss_Record_Share_%
0,Bookcases,0.00,31935.98,6075.71,60,19.02,0.00
1,Bookcases,0.15,27558.52,1418.99,52,5.15,32.69
2,Bookcases,0.20,26841.50,130.50,46,0.49,50.00
3,Bookcases,0.30,4282.70,-555.87,10,-12.98,90.00
4,Bookcases,0.32,14493.46,-2391.14,27,-16.50,100.00
5,Bookcases,0.50,7308.46,-4255.81,18,-58.23,100.00
6,Bookcases,0.70,2459.38,-3894.94,15,-158.37,100.00
7,Tables,0.00,71578.76,13276.30,72,18.55,0.00
8,Tables,0.20,45430.23,-303.56,71,-0.67,43.66
9,Tables,0.30,25182.15,-3402.33,54,-13.51,92.59


Preliminary finding

Discount-driven margin erosion
Both Tables and Bookcases are profitable in the observed zero-discount groups but become unprofitable at higher discount levels.




Product-specific exposure
Tables show negative aggregate profit at 20% discount, while Bookcases retain a small positive margin at that level.




Potential policy issue
Large discounts may be destroying margin without generating enough additional profit to compensate. We still need to examine whether the discount levels reflect different products, order sizes, or customer types.

### Are losses concentrated in particular product orders, or are they spread broadly across Tables and Bookcases?

In [4]:
loss_orders = df[
    (df["Sub-Category"].isin(["Tables", "Bookcases"])) &
    (df["Profit"] < 0)
].copy()

loss_orders = loss_orders.sort_values("Profit").head(15)

display(
    loss_orders[
        [
            "Order ID",
            "Order Date",
            "Region",
            "State",
            "Sub-Category",
            "Product Name",
            "Sales",
            "Quantity",
            "Discount",
            "Profit"
        ]
    ].round(2)
)

,Order ID,Order Date,Region,State,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
9639,CA-2015-116638,1/28/2015,South,North Carolina,Tables,Chromcraft Bull-Nose Wood Oval Conference Tabl...,4297.64,13,0.4,-1862.31
27,US-2015-150630,9/17/2015,East,Pennsylvania,Bookcases,"Riverside Palais Royal Lawyers Bookcase, Royal...",3083.43,7,0.5,-1665.05
5320,US-2017-162558,10/2/2017,South,Tennessee,Tables,Chromcraft Bull-Nose Wood Oval Conference Tabl...,2314.12,7,0.4,-1002.78
8993,CA-2017-159100,10/16/2017,South,Tennessee,Tables,Balt Solid Wood Round Tables,1875.26,7,0.4,-968.88
463,CA-2016-109869,4/22/2016,West,Arizona,Tables,Bush Advantage Collection Racetrack Conference...,1272.63,6,0.5,-814.48
1369,US-2015-103471,12/24/2015,West,Colorado,Bookcases,"Atlantic Metals Mobile 4-Shelf Bookcases, Cust...",590.06,7,0.7,-786.74
6639,CA-2014-108609,10/31/2014,East,Ohio,Tables,Hon 94000 Series Round Tables,1421.66,8,0.4,-734.53
5066,CA-2017-142090,11/30/2017,South,North Carolina,Tables,Bush Advantage Collection Racetrack Conference...,1781.68,7,0.4,-653.28
949,US-2017-110576,11/28/2017,East,Pennsylvania,Tables,"Riverside Furniture Oval Coffee Table, Oval En...",2065.32,12,0.4,-619.60
1713,US-2017-124968,9/8/2017,Central,Illinois,Tables,BoxOffice By Design Rectangular and Half-Moon ...,765.62,7,0.5,-566.56


Finding 1: Tables dominate the largest losses
In the 15 worst records you shared, 12 are Tables and 3 are Bookcases.
The largest loss is:
- Product: Chromcraft Bull-Nose Wood Oval Conference Table
- Region: South
- State: North Carolina
- Quantity: 13
- Sales: $4,297.64
- Discount: 40%
- Profit: −$1,862.31
This is a substantial loss on a single record. It suggests that heavily discounted table orders deserve closer investigation.

Finding 2: The issue appears across multiple regions
Loss-making table orders in your list appear in the South, East, West, and Central regions.

Finding 3: Several losses involve multiple units
The worst records often involve quantities of 6–13 units. That matters because a poor margin on a multi-unit order can create a substantial total loss.
However, we cannot conclude that large quantities cause losses from these 15 records alone. We selected these records specifically because their losses were the largest, so this is not a representative sample.

## Test whether a few specific products drive the losses<br>
Now we should investigate the full dataset, not just the 15 worst records.<br>
Our question is: Are certain table and bookcase models repeatedly generating losses, or is the problem mainly the discounting strategy across many products?

In [5]:
product_loss_analysis = (
    df[df["Sub-Category"].isin(["Tables", "Bookcases"])]
    .groupby(["Sub-Category", "Product Name"])
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Records=("Profit", "size"),
        Loss_Making_Records=("Profit", lambda x: (x < 0).sum()),
        Avg_Discount=("Discount", "mean")
    )
    .reset_index()
)

product_loss_analysis["Profit_Margin_%"] = (
    product_loss_analysis["Profit"]
    / product_loss_analysis["Sales"] * 100
)

product_loss_analysis["Loss_Record_Share_%"] = (
    product_loss_analysis["Loss_Making_Records"]
    / product_loss_analysis["Records"] * 100
)

# Focus on products with at least 5 records
product_loss_analysis = product_loss_analysis[
    product_loss_analysis["Records"] >= 5
]

# Show the 15 products with the lowest total profit
worst_products = product_loss_analysis.sort_values(
    "Profit"
).head(15)

display(
    worst_products[
        [
            "Sub-Category",
            "Product Name",
            "Sales",
            "Profit",
            "Records",
            "Avg_Discount",
            "Profit_Margin_%",
            "Loss_Record_Share_%"
        ]
    ].round(2)
)    

,Sub-Category,Product Name,Sales,Profit,Records,Avg_Discount,Profit_Margin_%,Loss_Record_Share_%
78,Tables,Chromcraft Bull-Nose Wood Oval Conference Tabl...,9917.64,-2876.12,5,0.28,-29.00,80.00
72,Tables,Bush Advantage Collection Racetrack Conference...,9544.72,-1934.40,7,0.35,-20.27,85.71
101,Tables,"Riverside Furniture Oval Coffee Table, Oval En...",4446.18,-1147.40,5,0.30,-25.81,80.00
25,Bookcases,O'Sullivan 4-Shelf Bookcase in Odessa Pine,2740.20,-975.10,7,0.42,-35.58,71.43
60,Tables,"Bevis Oval Conference Table, Walnut",6942.07,-856.01,8,0.28,-12.33,75.00
51,Tables,BPI Conference Tables,2241.87,-795.97,5,0.37,-35.50,80.00
34,Bookcases,O'Sullivan Plantations 2-Door Library in Landv...,2946.37,-767.74,5,0.33,-26.06,100.00
32,Bookcases,O'Sullivan Living Dimensions 5-Shelf Bookcases,3420.77,-755.75,6,0.24,-22.09,100.00
89,Tables,Hon 94000 Series Round Tables,7404.50,-681.21,5,0.16,-9.20,60.00
35,Bookcases,"Riverside Palais Royal Lawyers Bookcase, Royal...",15610.97,-669.54,5,0.23,-4.29,40.00


### Product-Level Root-Cause Analysis

**Objective:** Identify whether profitability losses in the Tables and Bookcases subcategories are concentrated in specific products or reflect a broader pattern.

#### Key Findings

**1. Tables account for most products among the worst-performing items.**

Among the 15 lowest-profit products with at least five records, 11 were Tables and 4 were Bookcases. The Chromcraft Bull-Nose Wood Oval Conference Table recorded the lowest aggregate profit at **−$2,876.12** across five records, with an average discount of 28% and a profit margin of −29.00%. The Bush Advantage Collection Racetrack Conference Table recorded **−$1,934.40** across seven records, with an average discount of 35% and a profit margin of −20.27%.

**2. Several Bookcases products also exhibit substantial losses.**

The O'Sullivan 4-Shelf Bookcase in Odessa Pine recorded aggregate profit of **−$975.10** across seven records. Its average discount was 42%, its profit margin was −35.58%, and 71.43% of its records were loss-making.

**3. High discounts are associated with losses, but they do not fully explain the pattern.**

The Hon 94000 Series Round Tables recorded aggregate profit of **−$681.21**, despite an average discount of only 16%. Its profit margin was −9.20%, and 60% of its records were loss-making. This indicates that the investigation should consider product-specific economics and order characteristics in addition to discounting.

**4. Losses recur across multiple products.**

Several products had loss-making shares of 75% or more, including some with loss-making records in every observed transaction. This suggests that the issue is not limited to a single unusually unprofitable order. However, because the analysis focused on the 15 lowest-profit products, it does not establish the prevalence of unprofitability across all products in these subcategories.

#### Preliminary Business Implication

The findings support investigating **product-specific pricing and discount practices**, particularly for poorly performing Tables products, while also examining Bookcases products with persistently negative margins. The analysis has not yet established whether discounts, product costs, or order composition are the primary drivers of these losses.

**Next step:** Quantify the overall profitability of Tables and Bookcases and measure how much of the negative profit is concentrated in the five worst-performing products before estimating the potential opportunity for intervention.

## Quantify the opportunity before recommending action

In [6]:
# Aggregate total profitability for Tables and Bookcases
target_products = df[
    df["Sub-Category"].isin(["Tables", "Bookcases"])
].copy()

subcategory_summary = (
    target_products
    .groupby("Sub-Category")
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Records=("Profit", "size"),
        Loss_Making_Records=("Profit", lambda x: (x < 0).sum())
    )
    .reset_index()
)

subcategory_summary["Profit_Margin_%"] = (
    subcategory_summary["Profit"]
    / subcategory_summary["Sales"] * 100
)

subcategory_summary["Loss_Record_Share_%"] = (
    subcategory_summary["Loss_Making_Records"]
    / subcategory_summary["Records"] * 100
)

display(subcategory_summary.round(2))

# How concentrated are losses across individual products?
product_summary = (
    target_products
    .groupby(["Sub-Category", "Product Name"])
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Records=("Profit", "size")
    )
    .reset_index()
)

loss_products = product_summary[
    product_summary["Profit"] < 0
].copy()

total_negative_product_profit = loss_products["Profit"].sum()

top_5_loss_products = loss_products.nsmallest(5, "Profit").copy()

top_5_loss_products["Share_of_Negative_Product_Profit_%"] = (
    top_5_loss_products["Profit"].abs()
    / abs(total_negative_product_profit) * 100
)

display(
    top_5_loss_products[
        [
            "Sub-Category",
            "Product Name",
            "Sales",
            "Profit",
            "Records",
            "Share_of_Negative_Product_Profit_%"
        ]
    ].round(2)
)

print("Total profit across all loss-making products:",
      round(total_negative_product_profit, 2))

print("Total profit across Tables and Bookcases:",
      round(target_products["Profit"].sum(), 2))

,Sub-Category,Sales,Profit,Records,Loss_Making_Records,Profit_Margin_%,Loss_Record_Share_%
0,Bookcases,114880.00,-3472.56,228,109,-3.02,47.81
1,Tables,206965.53,-17725.48,319,203,-8.56,63.64


,Sub-Category,Product Name,Sales,Profit,Records,Share_of_Negative_Product_Profit_%
78,Tables,Chromcraft Bull-Nose Wood Oval Conference Tabl...,9917.64,-2876.12,5,10.08
72,Tables,Bush Advantage Collection Racetrack Conference...,9544.72,-1934.40,7,6.78
53,Tables,Balt Solid Wood Round Tables,6518.75,-1201.06,4,4.21
67,Tables,BoxOffice By Design Rectangular and Half-Moon ...,1706.25,-1148.44,3,4.03
101,Tables,"Riverside Furniture Oval Coffee Table, Oval En...",4446.18,-1147.40,5,4.02


Total profit across all loss-making products: -28525.18
Total profit across Tables and Bookcases: -21198.04


### Quantifying the Profitability Problem

The subcategory-level analysis identified Tables and Bookcases as two loss-making areas. Tables generated $206,965.53 in sales but recorded a net loss of $17,725.48, corresponding to a profit margin of −8.56%. Bookcases generated $114,880.00 in sales and recorded a net loss of $3,472.56, with a margin of −3.02%.

The frequency of loss-making records was also higher for Tables (63.64%) than for Bookcases (47.81%), making Tables the first priority for further investigation.

At the product level, the five products with the largest negative total profits accounted for approximately 29.1% of the negative profit summed across all loss-making products. The remaining losses were distributed across other products, suggesting that targeted product-level interventions may help but are unlikely to address the entire problem on their own.

**Business implication:** Prioritize an investigation of Table pricing and discount practices, while reviewing persistently unprofitable Bookcases products. Before recommending discount caps or estimating financial benefits, investigate whether the observed losses are associated with discount levels, particular products, regions, or order characteristics. These analyses are descriptive and do not establish causal effects.